# **Last.fm tags: feasibility check**

The main analysis (`moodtape_analysis.ipynb`) showed that tracks close on audio features can still
feel like a completely different vibe: audio features describe how a track sounds, not its style or
how listeners perceive it.

[Last.fm](https://www.last.fm) users tag tracks and artists with free-text labels ("indie",
"dream pop", "melancholic"…). This notebook checks, on a sample of the catalog, whether those tags
could add that missing context:

1. **Coverage:** how many of our tracks have tags, at track level and at artist level?
2. **Content:** do the tags describe mood, style, or only repeat genre information we already have?
3. **Usefulness:** do tags separate "Brooklyn Baby" from its audio-feature neighbours?

**Terms of use (as of November 2026):** the Last.fm API is free for non-commercial use, stored
Last.fm data is capped at 100 MB, responses must be cached, and Last.fm must be credited. API
responses are cached in `data/lastfm_cache.json` (git-ignored), so re-running this notebook does
not call the API again for requests already made.

In [ ]:
import json
import time
import urllib.error
import urllib.parse
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path('data')
tracks = pd.read_parquet(DATA_DIR / 'tracks_clean.parquet')


def read_env(path='../.env'):
    """Minimal .env reader (KEY=value lines), to avoid an extra dependency."""
    env = {}
    for line in Path(path).read_text().splitlines():
        if '=' in line and not line.lstrip().startswith('#'):
            key, value = line.split('=', 1)
            env[key.strip()] = value.strip()
    return env


API_KEY = read_env().get('LASTFM_API_KEY')
assert API_KEY, "Set LASTFM_API_KEY in the .env file at the project root"
print(f"{len(tracks):,} tracks loaded")

## **Sample**

300 tracks, 75 from each popularity band. Less popular tracks probably have fewer tags, so a purely
random sample (dominated by mid-popularity tracks) would hide how coverage changes along the catalog.

In [ ]:
POPULARITY_BANDS = [-1, 9, 29, 49, 100]
BAND_LABELS = ['0-9', '10-29', '30-49', '50-100']
tracks['popularity_band'] = pd.cut(tracks['popularity'], POPULARITY_BANDS, labels=BAND_LABELS)

sample = (
    tracks.groupby('popularity_band', observed=True)
    .sample(n=75, random_state=42)
    .reset_index(drop=True)
)
sample['artist'] = sample['artists'].str[0]   # first credited artist
sample['popularity_band'].value_counts().sort_index()

## **Fetching tags**

Two endpoints, both looked up by name (Last.fm has no Spotify IDs), with autocorrection of small
spelling differences:

* `track.getTopTags`: tags for a specific track (artist + track name);
* `artist.getTopTags`: tags for an artist (first credited artist).

Each tag comes with a relative weight from 0 to 100 (100 = the most used tag for that track or artist).
Tags below `MIN_WEIGHT` are ignored as noise.

In [ ]:
CACHE_PATH = DATA_DIR / 'lastfm_cache.json'
cache = json.loads(CACHE_PATH.read_text(encoding='utf-8')) if CACHE_PATH.exists() else {}
MIN_WEIGHT = 10


def call_api(**params):
    """Cached Last.fm API call. Returns the parsed JSON (API errors included)."""
    cache_key = json.dumps(params, sort_keys=True, ensure_ascii=False)
    if cache_key not in cache:
        query = urllib.parse.urlencode({**params, 'autocorrect': 1, 'api_key': API_KEY, 'format': 'json'})
        try:
            with urllib.request.urlopen(f'https://ws.audioscrobbler.com/2.0/?{query}', timeout=20) as response:
                cache[cache_key] = json.load(response)
        except urllib.error.HTTPError as error:
            cache[cache_key] = json.load(error)   # API errors come back as JSON too
        time.sleep(0.25)                           # stay well below the API rate limit
    return cache[cache_key]


def parse_tags(data):
    """[(tag, weight)] with weight >= MIN_WEIGHT, or None if Last.fm does not know the item."""
    if 'error' in data:
        return None
    tags = data.get('toptags', {}).get('tag', [])
    return [(t['name'].lower(), int(t['count'])) for t in tags if int(t['count']) >= MIN_WEIGHT]


def track_tags(artist, track):
    return parse_tags(call_api(method='track.getTopTags', artist=artist, track=track))


def artist_tags(artist):
    return parse_tags(call_api(method='artist.getTopTags', artist=artist))


def save_cache():
    CACHE_PATH.write_text(json.dumps(cache, ensure_ascii=False), encoding='utf-8')


sample['track_tags'] = [track_tags(a, t) for a, t in zip(sample['artist'], sample['track_name'])]
sample['artist_tags'] = [artist_tags(a) for a in sample['artist']]
save_cache()

print(f"Cache: {len(cache)} responses, {CACHE_PATH.stat().st_size / 1e6:.2f} MB")

## **1. Coverage**

In [ ]:
def n_tags(tags):
    return len(tags) if tags else 0


for level in ('track', 'artist'):
    sample[f'{level}_found'] = sample[f'{level}_tags'].notna()
    sample[f'{level}_n'] = sample[f'{level}_tags'].apply(n_tags)

coverage = sample.groupby('popularity_band', observed=True).agg(
    track_found=('track_found', 'mean'),
    track_3plus_tags=('track_n', lambda n: (n >= 3).mean()),
    artist_found=('artist_found', 'mean'),
    artist_3plus_tags=('artist_n', lambda n: (n >= 3).mean()),
)
coverage.loc['all'] = [
    sample['track_found'].mean(), (sample['track_n'] >= 3).mean(),
    sample['artist_found'].mean(), (sample['artist_n'] >= 3).mean(),
]
print("Share of sampled tracks (columns ending in 3plus_tags: at least 3 tags with weight >= MIN_WEIGHT):")
display((coverage * 100).round(0).astype(int).astype(str) + '%')

## **2. What do the tags describe?**

The most common artist tags in the sample, split into three groups:

* **genre:** tags that match one of the dataset's genre labels, i.e. information we already have;
* **mood:** tags from a hand-picked list of mood and atmosphere words;
* **other:** everything else: finer styles ("dream pop", "future bass"), decades, countries,
  vocals ("female vocalists"), opinions ("favorites")…

The mood list is a judgement call, written before looking at the results, so the split is indicative.

In [ ]:
dataset_genres = {g for genres in tracks['genres'] for g in genres}
dataset_genres |= {g.replace('-', ' ') for g in dataset_genres}

MOOD_WORDS = {
    'sad', 'melancholic', 'melancholy', 'happy', 'chill', 'chillout', 'relaxing', 'relax', 'calm',
    'mellow', 'energetic', 'upbeat', 'dark', 'dreamy', 'atmospheric', 'romantic', 'love', 'beautiful',
    'emotional', 'nostalgic', 'uplifting', 'aggressive', 'angry', 'party', 'feel good', 'fun',
    'sexy', 'summer', 'night', 'epic', 'haunting', 'ethereal', 'peaceful', 'depressing', 'groovy',
    'catchy', 'soft', 'heartbreak', 'lonely', 'hopeful', 'intense', 'smooth', 'sleep', 'study',
}


def tag_kind(tag):
    if tag in MOOD_WORDS:
        return 'mood'
    if tag in dataset_genres:
        return 'genre'
    return 'other'


all_artist_tags = [tag for tags in sample['artist_tags'].dropna() for tag, _ in tags]
top_tags = pd.DataFrame(Counter(all_artist_tags).most_common(40), columns=['tag', 'tracks'])
top_tags['kind'] = top_tags['tag'].apply(tag_kind)
display(top_tags)

kinds = pd.Series([tag_kind(t) for t in all_artist_tags]).value_counts(normalize=True)
print("Share of all artist tags by kind:", (kinds * 100).round(0).astype(int).to_dict())

has_mood = sample['artist_tags'].apply(lambda tags: any(tag_kind(t) == 'mood' for t, _ in tags) if tags else False)
has_new_info = sample['artist_tags'].apply(lambda tags: any(tag_kind(t) != 'genre' for t, _ in tags) if tags else False)
print(f"Tracks whose artist has at least one mood tag: {has_mood.mean():.0%}")
print(f"Tracks whose artist has at least one tag beyond the dataset genres: {has_new_info.mean():.0%}")

## **3. Do tags separate "Brooklyn Baby" from its audio neighbours?**

The 10 nearest neighbours of "Brooklyn Baby" on the 9 audio features (from the main analysis)
included dubstep, emo and show tunes. If tags capture what audio features miss, those neighbours
should share few tags with it. Overlap is measured with the Jaccard index (shared tags / all tags)
on artist tags. As a contrast, the same is computed for other Lana Del Rey tracks and for tracks by
artists Last.fm considers similar in style.

In [ ]:
from sklearn.neighbors import NearestNeighbors

vectors = np.stack(tracks['features_scaled'].to_numpy())
is_lana = tracks['artists'].apply(lambda a: 'Lana Del Rey' in a)
ref_idx = tracks.index[(tracks['track_name'] == 'Brooklyn Baby') & is_lana][0]
_, neighbour_idx = NearestNeighbors().fit(vectors).kneighbors(vectors[[ref_idx]], n_neighbors=11)


def tag_set(artist):
    return {tag for tag, _ in (artist_tags(artist) or [])}


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 0.0


ref_tags = tag_set('Lana Del Rey')
print("Lana Del Rey tags:", sorted(ref_tags))

rows = []
for idx in neighbour_idx[0][1:]:
    artist = tracks.at[idx, 'artists'][0]
    tags = tag_set(artist)
    rows.append({'track_name': tracks.at[idx, 'track_name'], 'artist': artist,
                 'shared_tags': ', '.join(sorted(ref_tags & tags)), 'jaccard': jaccard(ref_tags, tags)})
save_cache()

neighbours = pd.DataFrame(rows)
display(neighbours.round(2))
print(f"Mean Jaccard, audio neighbours: {neighbours['jaccard'].mean():.2f}")

# Contrast: artists Last.fm lists as similar to Lana Del Rey that are in our catalog
similar = call_api(method='artist.getSimilar', artist='Lana Del Rey', limit=50)
similar_names = [a['name'] for a in similar.get('similarartists', {}).get('artist', [])]
in_catalog = [name for name in similar_names if tracks['artists'].apply(lambda a: name in a).any()][:10]
contrast = pd.DataFrame({'artist': in_catalog,
                         'jaccard': [jaccard(ref_tags, tag_set(a)) for a in in_catalog]})
save_cache()
display(contrast.round(2))
print(f"Mean Jaccard, similar artists in the catalog: {contrast['jaccard'].mean():.2f}")

## **Findings**

Results on the 300-track sample (numbers from the cells above):

**1. Track-level tags are not usable.** Last.fm finds almost every track (99%), but only 2% have at
least 3 tags. Track tags exist only for very popular tracks: "Creep" (Radiohead, ~4 million
listeners) has them, while "Car Wash" (Rose Royce, ~200 thousand listeners) has none. Even among the
most popular band (popularity 50–100), only 8% of tracks have 3+ tags.

**2. Artist-level tags have good coverage.** Every sampled first artist was found, and 81% have at
least 3 tags, with little difference between popularity bands (77–85%). The trade-off is
granularity: all tracks by an artist share the same tags.

**3. Tags describe style, not mood.** Almost half of the artist tags repeat genre labels we already
have (46%), and most of the rest are finer styles, decades, countries and vocal descriptors
("dream pop", "80s", "brazilian", "female vocalists"). Mood words are rare: 1% of tags, and only 4%
of tracks have an artist with any mood tag. Still, 87% of tracks get at least one tag beyond the
dataset's genre labels.

**4. Artist tags separate "Brooklyn Baby" from its audio neighbours.** The 10 nearest neighbours on
audio features share very few tags with Lana Del Rey (mean Jaccard 0.06), while artists that
Last.fm lists as similar to her share many more (0.47). Caveat: Last.fm's similar artists are
partly derived from the same listener data as tags, so this contrast flatters tags somewhat; it
shows that tags carry information audio features lack, not how good a tag-based ranking would be.

**Conclusion:** Last.fm does not provide mood information for this catalog, but artist tags are a
reliable, broad source of *style* context, which is exactly what audio features miss. Combining
both (sound from audio features, style from artist tags) looks feasible.

**Other caveats:**
* Artists are matched by name with autocorrection, so artists with common or ambiguous names may be
  matched to the wrong Last.fm artist. This was not measured here.
* `artist.getSimilar` (used above for the contrast) is another possible signal: it reflects
  listening behaviour, which neither the dataset nor tags capture directly.
* Fetching artist tags for the whole catalog means ~17,600 requests (unique first artists), about
  1–2 hours at the request rate used here, well within the 100 MB storage cap.